# Modeling.

We train **six models**: three architectures × two feature sets.

| | price only (`close`) | price + volume (`close_volume`) |
|---|---|---|
| **LSTM** (main model) | A | B |
| **GRU** | C | D |
| **1D-CNN** | E | F |

Each run saves its model, scalers and metadata to `artifacts/<run>/`. We judge the runs on the **validation** set here; the test set is kept for the next notebook.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import config

Next: [`04_evaluation.ipynb`](04_evaluation.ipynb)

In [2]:
import pandas as pd
import numpy as np

from src import features

raw_data = pd.read_csv(
    config.DATA_PATH,
    header=[0, 1],
    skiprows=[2],
    index_col=0,
    parse_dates=True
)

aapl_data = raw_data.xs(config.TICKER, axis=1, level=1)

data = features.build_modeling_frame(aapl_data)

print(data.shape)
print(data.head())

(4209, 11)
Price          Close      High       Low      Open     Volume  log_return  \
2010-01-05  6.412026  6.448217  6.378228  6.418606  601904800    0.001727   
2010-01-06  6.310036  6.437451  6.303456  6.412028  552160000   -0.016034   
2010-01-07  6.298370  6.340842  6.252608  6.333365  477131200   -0.001851   
2010-01-08  6.340245  6.340843  6.252909  6.289997  447610800    0.006627   
2010-01-11  6.284315  6.370753  6.234664  6.364771  462229600   -0.008861   

Price       log_volume  volume_change  next_close target_date    target  
2010-01-05   20.215610       0.198111    6.310036  2010-01-06  0.001727  
2010-01-06   20.129348      -0.086261    6.298370  2010-01-07 -0.016034  
2010-01-07   19.983302      -0.146046    6.340245  2010-01-08 -0.001851  
2010-01-08   19.919435      -0.063867    6.284315  2010-01-11  0.006627  
2010-01-11   19.951572       0.032138    6.212829  2010-01-12 -0.008861  


In [3]:
from sklearn.preprocessing import StandardScaler

n = len(data)

train_end = int(n * config.TRAIN_RATIO)
val_end = train_end + int(n * config.VAL_RATIO)

train_data = data.iloc[:train_end].copy()
val_data = data.iloc[train_end:val_end].copy()
test_data = data.iloc[val_end:].copy()

feature_cols = ["log_return", "log_volume", "volume_change"]

scaler = StandardScaler()

train_scaled = train_data.copy()
val_scaled = val_data.copy()
test_scaled = test_data.copy()

train_scaled[feature_cols] = scaler.fit_transform(train_data[feature_cols])
val_scaled[feature_cols] = scaler.transform(val_data[feature_cols])
test_scaled[feature_cols] = scaler.transform(test_data[feature_cols])

print("Train:", train_scaled.shape)
print("Validation:", val_scaled.shape)
print("Test:", test_scaled.shape)

Train: (2946, 11)
Validation: (631, 11)
Test: (632, 11)


In [4]:
lookback = config.LOOKBACK

def make_windows(df, feature_cols, lookback):
    X = []
    y = []

    for i in range(lookback, len(df)):
        X.append(df[feature_cols].iloc[i-lookback:i].values)
        y.append(df["target"].iloc[i])

    return np.array(X), np.array(y)

X_train, y_train = make_windows(train_scaled, feature_cols, lookback)
X_val, y_val = make_windows(val_scaled, feature_cols, lookback)
X_test, y_test = make_windows(test_scaled, feature_cols, lookback)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (2886, 60, 3)
X_val: (571, 60, 3)
X_test: (572, 60, 3)


In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(config.SEED)

model_b = keras.Sequential([
    layers.Input(shape=(X_train.shape[1], X_train.shape[2])),
    layers.LSTM(64),
    layers.Dropout(config.DROPOUT),
    layers.Dense(1)
])

model_b.compile(
    optimizer=keras.optimizers.Adam(learning_rate=config.LEARNING_RATE),
    loss="mse",
    metrics=["mae"]
)

model_b.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,473 (68.25 KB)

 Trainable params: 17,473 (68.25 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=config.PATIENCE,
    restore_best_weights=True
)

history_b = model_b.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0030 - mae: 0.0405 - val_loss: 3.8348e-04 - val_mae: 0.0148
Epoch 2/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0011 - mae: 0.0255 - val_loss: 4.9832e-04 - val_mae: 0.0180
Epoch 3/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 7.3632e-04 - mae: 0.0203 - val_loss: 3.7457e-04 - val_mae: 0.0149
Epoch 4/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 5.8448e-04 - mae: 0.0180 - val_loss: 4.6379e-04 - val_mae: 0.0170
Epoch 5/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 5.2531e-04 - mae: 0.0167 - val_loss: 4.0991e-04 - val_mae: 0.0158
Epoch 6/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 4.4724e-04 - mae: 0.0156 - val_loss: 4.2790e-04 - val_mae: 0.0162
Epoch 7/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 4.0088e-04 - mae: 0.0147 - val_loss: 4.3581e-04 - val_mae: 0.0165
Epoch 8/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 3.9200e-04 - mae: 0.0145 - val_loss: 3.8964e-04 - val_mae: 0.0153
Epoch 9/

In [7]:
val_loss, val_mae = model_b.evaluate(X_val, y_val, verbose=0)
val_rmse = np.sqrt(val_loss)

print("Validation MAE:", val_mae)
print("Validation RMSE:", val_rmse)

Validation MAE: 0.014892838895320892
Validation RMSE: 0.019353807084020406


In [8]:
from pathlib import Path

artifact_dir = PROJECT_ROOT / "artifacts" / "model_b"
artifact_dir.mkdir(parents=True, exist_ok=True)

model_b.save(artifact_dir / "model_b.keras")

print("Saved to:", artifact_dir / "model_b.keras")

Saved to: /opt/aima-python/Stock_Price_DL/artifacts/model_b/model_b.keras


In [9]:
import joblib
import json

joblib.dump(scaler, artifact_dir / "scaler.pkl")

metadata = {
    "model": "LSTM",
    "run": "B",
    "features": feature_cols,
    "lookback": lookback,
    "validation_mae": float(val_mae),
    "validation_rmse": float(val_rmse)
}

with open(artifact_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Scaler and metadata saved.")

Scaler and metadata saved.
